# Portfolio Positions

This notebook builds one consolidated position for each asset across all accounts. Purchases increase quantity and invested capital, sales decrease current quantity, settlement transfers follow their credit/debit direction, and custody transfers are globally neutral. Dividends, interest, and income are excluded. The computed positions are saved under `data/store/computed_positions`.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from src.core.repositories.asset_repository import AssetRepository
from src.core.repositories.position_repository import PositionRepository
from src.core.repositories.transaction_repository import TransactionRepository
from src.core.services.position_service import PositionService

In [ ]:
transaction_repository = TransactionRepository(
    project_root / 'data/store/transactions/transactions.csv'
)
position_repository = PositionRepository(
    project_root / 'data/store/computed_positions/positions.csv'
)
asset_repository = AssetRepository(
    project_root / 'data/store/assets/assets.csv'
)

position_service = PositionService(
    transaction_repository=transaction_repository,
    position_repository=position_repository,
    asset_repository=asset_repository,
)
positions = position_service.build_and_save()
positions = position_service.all()

print(f'Computed and saved {len(positions)} positions.')

In [ ]:
import pandas as pd
from IPython.display import display

position_table = pd.DataFrame([
    {
        'asset_id': position.asset_id,
        'asset_name': position.asset.asset_name,
        'current_quantity': position.current_quantity,
        'total_acquired_quantity': position.total_acquired_quantity,
        'total_sold_quantity': position.total_sold_quantity,
        'invested_capital': position.invested_capital,
    }
    for position in positions
]).sort_values(['asset_id']).reset_index(drop=True)

display(position_table)

In [ ]:
# Confirm that the persisted file can be read back as typed Position objects.
saved_positions = position_service.all()
assert saved_positions == positions
print(f'Verified {len(saved_positions)} persisted positions.')